# Moondream Binary Classification

## Setup and Imports

In [1]:
import os
os.environ['HF_HOME'] = '../cache'

In [2]:
from PIL import Image
import joblib

from nazi_symbols_classification.training.data_preparation import get_image_paths
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

## Data Preparation

In [3]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection", ("train", "test", "val"))

In [4]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/val')]

In [5]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

## Model Loading and Classification

In [7]:
model = AutoModelForCausalLM.from_pretrained(
"vikhyatk/moondream2",
revision="2025-01-09",
trust_remote_code=True, # Uncomment for GPU acceleration & pip install accelerate # device_map={"": "cuda"}
device_map={"": "cuda"}
)

2025-06-18 10:36:24.755354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [22]:
def classify_document(doc_path, prompts):
    image = Image.open(doc_path)
    encoded_image = model.encode_image(image)

    for prompt in prompts:
        answer = model.query(encoded_image, prompt)["answer"]
        print(prompt, answer)
        if "Yes" in answer:
            return prompts[prompt]

    return "non-nazi"

In [9]:
prompts = {
    "Analyse whether the image contains a black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism?": "nazi",
    "Analyse whether the image contains the British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "nazi",
    "Analyse whether the image contains a broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "nazi",
    "Analyse whether the image is one of the historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "nazi",
    "Analyse whether the image is one of the images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "nazi",
    "Analyse whether the image is one of the images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "nazi",
    "Analyse whether the image is one of the images featuring the 'Happy Merchant' meme, a stereotypical representation of a smiling, hook-nosed Jewish figure used in online racist and antisemitic contexts.": "nazi",
    "Analyse whether the image is the imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "nazi",
    "Analyse whether the image contains a single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "nazi",
    "Analyse whether the image contains a skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "nazi",
    "Analyse whether a brownshirt (SA) symbol present in this image?": "nazi",
    "Analyse whether the image contains a black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "nazi",
    "Analyse whether the image contains the Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "nazi",
    "Analyse whether the image contains Nazi related symbols or insignia": "non-nazi",
}

In [27]:
classify_result = classify_document(test_images[6], prompts)
classify_result

Analyse whether the image contains a black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism?  No
Analyse whether the image contains the British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.  No
Analyse whether the image contains a broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.  No
Analyse whether the image is one of the historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.  No
Analyse whether the image is one of the images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.  No
Analyse whether the image is one of the images of the Judenstern, the yellow Star of David badge used durin

'nazi'

In [10]:
%%time

result = []

for image_path in test_images:
    result.append(classify_document(image_path, prompts))

CPU times: user 4h 30min 3s, sys: 1min 3s, total: 4h 31min 6s
Wall time: 3h 29min 59s


Save the results

In [13]:
to_store = dict(y_true=y_test, y_pred=result)

joblib.dump(to_store, "moondream-output/moondream_result_v2-binary.joblib")

['moondream-output/moondream_result_v2-binary.joblib']

Print the classification report and calculate roc_auc_score and accuracy_score

In [15]:
y_test, result = to_store["y_true"], to_store["y_pred"]

In [16]:
result = [int(label == "nazi") for label in result]
y_test_binary = [int(label == "nazi-symbol") for label in y_test]
print(classification_report(y_test_binary, result, digits=3))

              precision    recall  f1-score   support

           0      1.000     0.924     0.960     14813
           1      0.153     0.995     0.265       205

    accuracy                          0.925     15018
   macro avg      0.577     0.959     0.613     15018
weighted avg      0.988     0.925     0.951     15018



In [19]:
roc_auc_score(y_test_binary, result), accuracy_score(y_test_binary, result)

(0.9594863114633981, 0.9248235450792383)